# Overall outcomes 그림과 표 재생성
현재 프로젝트의 policy_summary.csv와 미시 low_income_summary.csv를 읽습니다.
패키지는 pandas, numpy, reportlab입니다. 없으면 터미널에서 `python -m pip install pandas numpy reportlab`을 실행하세요.
기본 가계 배분은 I이며 본문에서 다른 배분과 혼합하지 않습니다.
생성물은 노트북 실행 폴더의 overall_outcomes_exports에 저장됩니다.


In [ ]:
from pathlib import Path
import json, hashlib
import numpy as np
import pandas as pd
from reportlab.graphics.shapes import Drawing, String, Line, Rect, Polygon, Circle, Group
from reportlab.graphics.charts.lineplots import LinePlot
from reportlab.graphics.widgets.markers import makeMarker
from reportlab.graphics import renderPDF, renderSVG
from reportlab.lib import colors

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p/'Run_Master.gms').is_file())
OUT=PROJECT/'Figure'/'overall_outcomes_exports'
OUT.mkdir(parents=True,exist_ok=True)
DATA=PROJECT/'Figure'/'Sections_2_4'/'data'
s=pd.read_csv(DATA/'policy_summary.csv')
low=pd.read_csv(DATA/'low_income_summary.csv')
meta=json.loads((DATA/'cge_definitions.json').read_text(encoding='utf-8'))
factor=meta['model_money_to_billion_usd_conditional']
bau=s.loc[s.policy_id.eq('BAU')].iloc[0]
tou=s.loc[s.policy_id.eq('TOU')].iloc[0]
p=s.loc[s.rate.ne('base')&s.recycling_fraction.gt(0)].copy()
pol=pd.concat([s.loc[s.policy_id.eq('TOU')],p],ignore_index=True)
pol['CO2_reduction_Mt']=-pol.CO2_change_Mt
pol['SG_change_billion_USD']=(pol.SG-bau.SG)*factor
pol['fiscally_feasible']=pol.SG.ge(bau.SG-1e-10)
pol['loss_relief_pct']=(pol.EV-tou.EV)/(-tou.EV)*100
# Fix the micro allocation rule to I; pure REN and base cases use no cash allocation.
li=low.loc[low.low_income & low.allocation.isin(['I','none']),['policy_id','burden_increase_share']]
pol=pol.merge(li,on='policy_id',how='left')
pol['low_income_burden_increase_pct']=100*pol.burden_increase_share
ids=['TOU','r100_a00','r100_a50','r100_a100','r90_a60','r90_a100']
labels=dict(zip(ids,['TOU','H','A','B','C','D']))
pol['point_label']=pol.policy_id.map(labels)
table=pol.set_index('policy_id').loc[ids].reset_index()
table['r_pct']=100*table.recycling_fraction
table['REN_pct']=100*table.ren_fraction.fillna(0)
table['HH_pct']=100-table.REN_pct
cols=['point_label','policy_id','scenario','r_pct','REN_pct','HH_pct','EV','EV_PCT_CTH',
      'loss_relief_pct','CO2_reduction_Mt','SG_change_billion_USD','fiscally_feasible',
      'low_income_burden_increase_pct']
table[cols].to_csv(OUT/'Table_1_representative_policies.csv',index=False,encoding='utf-8-sig')
pol.to_csv(OUT/'Figure_1_policy_points.csv',index=False,encoding='utf-8-sig')
envelope=pol.loc[pol.rate.ne('base')].groupby('recycling_fraction').SG_change_billion_USD.agg(['min','max']).reset_index()
envelope.to_csv(OUT/'Figure_2_government_savings.csv',index=False,encoding='utf-8-sig')
manifest={'sources':{},'region':meta['region'],'micro_allocation':'I (none for no-HH cases)',
    'money_conversion_factor':factor,'policy_points_excluding_BAU':len(pol),
    'fiscally_feasible_points_including_TOU':int(pol.fiscally_feasible.sum()),
    'condition':'SG_policy >= SG_BAU','tolerance_for_numerical_comparison':1e-10}
for path in [DATA/'policy_summary.csv',DATA/'low_income_summary.csv']:
 manifest['sources'][str(path)]={'sha256':hashlib.sha256(path.read_bytes()).hexdigest()}
(OUT/'source_manifest.json').write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding='utf-8')

NAVY=colors.HexColor('#173B63');GRID=colors.HexColor('#E4E7EB');GREY=colors.HexColor('#52606D')
def Text(*args,**kwargs):
    kwargs.setdefault('fontName','Helvetica')
    return String(*args,**kwargs)

def shade(percent):
    start=np.array([225,238,250]);end=np.array([24,65,108])
    rgb=start+(end-start)*float(np.clip(percent/100,0,1))
    return colors.Color(*(rgb/255))
def setup_plot(width,height,x,y,w,h,xmin,xmax,ymin,ymax,xsteps,ysteps):
    d=Drawing(width,height)
    d.add(Rect(0,0,width,height,fillColor=colors.white,strokeColor=None))
    lp=LinePlot();lp.x=x;lp.y=y;lp.width=w;lp.height=h
    lp.xValueAxis.valueMin=xmin;lp.xValueAxis.valueMax=xmax;lp.xValueAxis.valueSteps=xsteps
    lp.yValueAxis.valueMin=ymin;lp.yValueAxis.valueMax=ymax;lp.yValueAxis.valueSteps=ysteps
    for ax in [lp.xValueAxis,lp.yValueAxis]:
        ax.labels.fontName='Helvetica';ax.labels.fontSize=9
        ax.strokeColor=GREY;ax.strokeWidth=.6
    lp.xValueAxis.labelTextFormat='%g'
    lp.yValueAxis.labelTextFormat='%.2f'
    lp.yValueAxis.visibleGrid=True;lp.yValueAxis.gridStrokeColor=GRID;lp.yValueAxis.gridStrokeWidth=.45
    def pos(xx,yy):return x+(xx-xmin)/(xmax-xmin)*w,y+(yy-ymin)/(ymax-ymin)*h
    return d,lp,pos

def overview():
    d,lp,pos=setup_plot(680,460,65,67,510,320,0,14,-.30,.01,[0,2,4,6,8,10,12,14],[-.30,-.25,-.20,-.15,-.10,-.05,0])
    lp.data=[[(r.CO2_reduction_Mt,r.EV_PCT_CTH)] for r in pol.itertuples()]+[[(0,0)]]
    for i,r in enumerate(pol.itertuples()):
        m=makeMarker('FilledCircle' if r.fiscally_feasible else 'Cross')
        m.size=5.4 if r.fiscally_feasible else 7.2
        m.fillColor=shade(r.low_income_burden_increase_pct)
        m.strokeColor=shade(r.low_income_burden_increase_pct);m.strokeWidth=.85
        lp.lines[i].symbol=m;lp.lines[i].strokeColor=None
    m=makeMarker('FilledDiamond');m.size=7;m.fillColor=GREY;m.strokeColor=GREY
    lp.lines[len(pol)].symbol=m;lp.lines[len(pol)].strokeColor=None
    d.add(lp)
    d.add(Text(65,435,'Overall policy outcomes',fontName='Helvetica-Bold',fontSize=13,fillColor=NAVY))
    d.add(Circle(70,414,2.8,fillColor=GREY,strokeColor=GREY))
    d.add(Text(80,411,'SG policy >= SG BAU',fontSize=9,fillColor=GREY))
    d.add(Line(254,414.5,261,414.5,strokeColor=GREY));d.add(Line(257.5,411,257.5,418,strokeColor=GREY))
    d.add(Text(270,411,'SG policy < SG BAU',fontSize=9,fillColor=GREY))
    d.add(Text(320,26,'CO2 reduction relative to BAU (Mt)',textAnchor='middle',fontSize=11))
    d.add(Group(Text(0,0,'EV / benchmark household consumption (%)',textAnchor='middle',fontSize=11),transform=(0,1,-1,0,17,240)))
    barx,bary,barw,barh=604,91,13,250
    for i in range(100):d.add(Rect(barx,bary+i*barh/100,barw,barh/100+.2,fillColor=shade(i),strokeColor=None))
    for v in [0,25,50,75,100]:
        yy=bary+barh*v/100
        d.add(Line(barx+barw,yy,barx+barw+3,yy,strokeColor=GREY,strokeWidth=.5))
        d.add(Text(barx+barw+6,yy-3,str(v),fontSize=9,fillColor=GREY))
    d.add(Group(Text(0,0,'Low-income households with higher net bills (%)',textAnchor='middle',fontSize=9),transform=(0,1,-1,0,664,216)))
    offsets={'TOU':(-32,-17),'H':(-30,13),'A':(-28,23),'B':(12,17),'C':(19,-23),'D':(-21,-31)}
    for r in table.itertuples():
        px,py=pos(r.CO2_reduction_Mt,r.EV_PCT_CTH)
        label=r.point_label;dx,dy=offsets[label]
        lx,ly=px+dx,py+dy
        d.add(Line(px,py,lx+(4 if dx<0 else 0),ly+(3 if dy<0 else -2),strokeColor=GREY,strokeWidth=.65))
        d.add(Text(lx,ly,label,fontName='Helvetica-Bold',fontSize=11,fillColor=colors.black))
    bx,by=pos(0,0);d.add(Text(bx+11,by-6,'BAU',fontSize=10,fillColor=GREY))
    return d

def fiscal():
    e=envelope.loc[envelope.recycling_fraction.ge(.7)]
    d,lp,pos=setup_plot(680,350,72,65,535,225,.7,1,-.16,.70,[.7,.8,.9,1],[-.1,0,.1,.2,.3,.4,.5,.6,.7])
    lp.xValueAxis.labelTextFormat=lambda v:f'{100*v:.0f}'
    points=[pos(r.recycling_fraction,r['min']) for _,r in e.iterrows()]
    points+=list(reversed([pos(r.recycling_fraction,r['max']) for _,r in e.iterrows()]))
    d.add(Polygon([v for point in points for v in point],fillColor=colors.HexColor('#DBE8F4'),strokeColor=None))
    lp.data=[list(zip(e.recycling_fraction,e['min'])),list(zip(e.recycling_fraction,e['max']))]
    for i,col in enumerate([NAVY,colors.HexColor('#6899BC')]):
        lp.lines[i].strokeColor=col;lp.lines[i].strokeWidth=1.1
        lp.lines[i].symbol=makeMarker('FilledCircle');lp.lines[i].symbol.size=5
        lp.lines[i].symbol.fillColor=col;lp.lines[i].symbol.strokeColor=col
    lp.lines[1].strokeDashArray=[3,2]
    d.add(lp)
    x0,y0=pos(.7,0);x1,_=pos(1,0)
    d.add(Line(x0,y0,x1,y0,strokeColor=colors.HexColor('#A44742'),strokeWidth=1,strokeDashArray=[4,3]))
    xx,ymin=pos(.9,-.16);_,ymax=pos(.9,.7)
    d.add(Line(xx,ymin,xx,ymax,strokeColor=GREY,strokeWidth=.65,strokeDashArray=[2,3]))
    d.add(Text(72,326,'Government-savings preservation near the boundary',fontName='Helvetica-Bold',fontSize=13,fillColor=NAVY))
    d.add(Text(72,306,'Envelope across REN allocations at each sampled recycling rate',fontSize=9,fillColor=GREY))
    d.add(Text(340,24,'Recycling share r (%)',textAnchor='middle',fontSize=11))
    d.add(Group(Text(0,0,'Change in government savings (USD billion)',textAnchor='middle',fontSize=10),transform=(0,1,-1,0,18,178)))
    d.add(Text(xx+7,278,'90%: last feasible sampled rate',fontSize=9,fillColor=GREY))
    return d

fig1=overview();fig2=fiscal()
for name,drawing in [('Figure_1_overall',fig1),('Figure_2_fiscal_condition',fig2)]:
 renderPDF.drawToFile(drawing,str(OUT/(name+'.pdf')))
 renderSVG.drawToFile(drawing,str(OUT/(name+'.svg')))
print('Policy points',len(pol),'feasible',pol.fiscally_feasible.sum())
print(table[['point_label','policy_id','EV_PCT_CTH','loss_relief_pct','CO2_reduction_Mt','SG_change_billion_USD']].to_string(index=False))


In [ ]:
from IPython.display import SVG, display
display(SVG(filename=str(OUT/'Figure_1_overall.svg')))
display(table[cols])
display(SVG(filename=str(OUT/'Figure_2_fiscal_condition.svg')))